# aloevera Demo

This notebook demonstrates all aloevera organizers:
- **`avr.tabs`** — horizontal tabs
- **`avr.accordion`** — collapsible sections
- **`avr.dropdown`** — dropdown selector
- **`avr.scroll`** — scrollable list
- **Nesting** — organizers inside organizers
- **Copy / Download** — export to HTML for Confluence etc.

Each organizer widget has small **Copy** and **Download** buttons on the right side.

In [ ]:
# Install dependencies (run once)
%pip install polars plotly ipywidgets -q

In [ ]:
import sys, os

# Make aloevera importable from the demo folder
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import aloevera as avr
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

print(f"aloevera {avr.__version__}")
print(f"polars {pl.__version__}")

## 1. Create sample data with Polars

In [ ]:
np.random.seed(42)

# Monthly sales data
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

df_sales = pl.DataFrame({
    "month": months,
    "revenue": np.random.randint(50_000, 150_000, 12).tolist(),
    "expenses": np.random.randint(30_000, 100_000, 12).tolist(),
    "customers": np.random.randint(200, 800, 12).tolist(),
})

# Time series data
df_ts = pl.DataFrame({
    "day": list(range(1, 91)),
    "price": np.cumsum(np.random.randn(90) * 2 + 0.1).tolist(),
    "volume": np.random.randint(100, 1000, 90).tolist(),
})

# Distribution data
df_dist = pl.DataFrame({
    "group_a": np.random.normal(50, 10, 200).tolist(),
    "group_b": np.random.normal(60, 15, 200).tolist(),
    "group_c": np.random.normal(45, 8, 200).tolist(),
})

print(f"Sales data: {df_sales.shape}")
print(f"Time series: {df_ts.shape}")
print(f"Distribution: {df_dist.shape}")
df_sales.head()

## 2. Build Plotly figures

In [ ]:
# Bar chart — revenue vs expenses
fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(
    x=df_sales["month"].to_list(),
    y=df_sales["revenue"].to_list(),
    name="Revenue",
    marker_color="#4285f4",
))
fig_bar.add_trace(go.Bar(
    x=df_sales["month"].to_list(),
    y=df_sales["expenses"].to_list(),
    name="Expenses",
    marker_color="#ea4335",
))
fig_bar.update_layout(
    title="Monthly Revenue vs Expenses",
    barmode="group",
    height=400,
    template="plotly_white",
)

# Line chart — price over time
fig_line = go.Figure()
fig_line.add_trace(go.Scatter(
    x=df_ts["day"].to_list(),
    y=df_ts["price"].to_list(),
    mode="lines",
    name="Price",
    line=dict(color="#34a853", width=2),
))
fig_line.update_layout(
    title="Price Over 90 Days",
    xaxis_title="Day",
    yaxis_title="Price",
    height=400,
    template="plotly_white",
)

# Histogram — distribution comparison
fig_hist = go.Figure()
for col, color, name in [
    ("group_a", "#4285f4", "Group A"),
    ("group_b", "#ea4335", "Group B"),
    ("group_c", "#fbbc04", "Group C"),
]:
    fig_hist.add_trace(go.Histogram(
        x=df_dist[col].to_list(),
        name=name,
        marker_color=color,
        opacity=0.7,
    ))
fig_hist.update_layout(
    title="Distribution Comparison",
    barmode="overlay",
    height=400,
    template="plotly_white",
)

# Scatter plot — customers vs revenue
fig_scatter = go.Figure()
fig_scatter.add_trace(go.Scatter(
    x=df_sales["customers"].to_list(),
    y=df_sales["revenue"].to_list(),
    mode="markers+text",
    text=df_sales["month"].to_list(),
    textposition="top center",
    marker=dict(size=12, color="#9c27b0"),
))
fig_scatter.update_layout(
    title="Customers vs Revenue",
    xaxis_title="Customers",
    yaxis_title="Revenue",
    height=400,
    template="plotly_white",
)

# Volume bar chart
fig_vol = go.Figure()
fig_vol.add_trace(go.Bar(
    x=df_ts["day"].to_list(),
    y=df_ts["volume"].to_list(),
    marker_color="#00bcd4",
))
fig_vol.update_layout(
    title="Daily Trading Volume",
    height=400,
    template="plotly_white",
)

print("Created 5 figures: fig_bar, fig_line, fig_hist, fig_scatter, fig_vol")

## 3. `avr.tabs` — Horizontal tabs

Switch between figures by clicking the tab headers. Notice the **Copy** and **Download** buttons on the right.

In [ ]:
avr.tabs(
    titles=["Revenue vs Expenses", "Price Trend", "Distributions"],
    contents=[fig_bar, fig_line, fig_hist],
)

## 4. `avr.accordion` — Collapsible sections

Click a section header to expand/collapse it.

In [ ]:
avr.accordion(
    titles=["Bar Chart", "Scatter Plot", "Volume"],
    contents=[fig_bar, fig_scatter, fig_vol],
)

## 5. `avr.dropdown` — Dropdown selector

Pick a figure from the dropdown menu.

In [ ]:
avr.dropdown(
    titles=["Revenue vs Expenses", "Price Trend", "Distributions", "Customers vs Revenue"],
    contents=[fig_bar, fig_line, fig_hist, fig_scatter],
)

## 6. `avr.scroll` — Scrollable list

All figures stacked vertically in a scrollable container.

In [ ]:
avr.scroll(
    titles=["Bar", "Line", "Histogram", "Scatter", "Volume"],
    contents=[fig_bar, fig_line, fig_hist, fig_scatter, fig_vol],
    height="500px",
)

## 7. HTML content — DataFrames as tables

You can pass DataFrames (converted to HTML) or raw HTML strings alongside Plotly figures.

In [ ]:
# Mix Plotly figures with DataFrame HTML and raw HTML
summary_html = "<h3>Summary</h3><p>This dashboard shows <b>monthly sales data</b> with revenue, expenses, and customer counts across 12 months.</p>"

avr.tabs(
    titles=["Chart", "Data Table", "Summary"],
    contents=[
        fig_bar,
        df_sales.to_pandas(),  # DataFrame -> HTML table automatically
        summary_html,          # raw HTML string
    ],
)

## 8. Nesting — Organizers inside organizers

Full composition: tabs containing an accordion, and a dropdown containing scrollable content.

In [ ]:
# Nested: top-level tabs containing different organizer types
charts_accordion = avr.accordion(
    titles=["Bar Chart", "Line Chart", "Histogram"],
    contents=[fig_bar, fig_line, fig_hist],
)

data_dropdown = avr.dropdown(
    titles=["Sales Table", "Time Series Table"],
    contents=[df_sales.to_pandas(), df_ts.to_pandas()],
)

# Top-level tabs wrapping the nested organizers
avr.tabs(
    titles=["Charts (Accordion)", "Data (Dropdown)", "Scatter"],
    contents=[charts_accordion, data_dropdown, fig_scatter],
)

## 9. Programmatic HTML export

You can also export HTML programmatically without using the buttons — useful for scripts or CI pipelines.

In [ ]:
# Every organizer widget carries _export_html for the raw fragment
widget = avr.tabs(
    titles=["Bar", "Line"],
    contents=[fig_bar, fig_line],
)

# Get the raw HTML fragment (paste into Confluence /html macro)
html_fragment = widget._export_html
print(f"HTML fragment length: {len(html_fragment)} chars")
print(f"First 200 chars:\n{html_fragment[:200]}...")

# Or wrap it in a full standalone page and save to disk
full_page = avr.make_standalone(html_fragment, title="My Dashboard")
with open("export_demo.html", "w") as f:
    f.write(full_page)

print(f"\nSaved standalone HTML: export_demo.html ({len(full_page)} chars)")

## 10. Edge cases

Test validation and mixed content types.

In [ ]:
# Test validation: mismatched lengths should raise ValueError
try:
    avr.tabs(titles=["A", "B"], contents=[fig_bar])
except ValueError as e:
    print(f"Caught expected error: {e}")

# Test validation: empty lists should raise ValueError
try:
    avr.accordion(titles=[], contents=[])
except ValueError as e:
    print(f"Caught expected error: {e}")

# Test validation: wrong types should raise TypeError
try:
    avr.dropdown(titles="not a list", contents="not a list")
except TypeError as e:
    print(f"Caught expected error: {e}")

print("\nAll validation tests passed!")